# Macaron art images
Makes photo-real images for the hot pot builder: two empty pots, broth surfaces and each ingredient on its own with the background removed. Everything is generated, so no one else's photos are used.

In [ ]:
# add only what is missing, without upgrading numpy (an upgrade mid-session breaks diffusers)
!pip -q install --no-deps rembg pymatting onnxruntime flatbuffers coloredlogs humanfriendly 2>&1 | tail -2
!python -c "import diffusers" 2>/dev/null || pip -q install --no-deps diffusers 2>&1 | tail -2
import numpy; print('numpy', numpy.__version__)

In [ ]:
import torch, os, io, time
from PIL import Image, ImageDraw
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
from rembg import remove, new_session
MODEL = "SG161222/RealVisXL_V4.0"
try: pipe = StableDiffusionXLPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
except Exception as e:
    print("no fp16 files, loading full:", str(e)[:80]); pipe = StableDiffusionXLPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, use_safetensors=True)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe = pipe.to("cuda")
try: pipe.vae.enable_slicing()
except Exception: pass
CUT = new_session("isnet-general-use")
NEG = "cartoon, illustration, drawing, anime, 3d render, cgi, painting, text, watermark, logo, hands, person, chopsticks, spoon, cutting board, table, blurry, lowres, deformed, oversaturated"
def gen(prompt, w, h, seed, steps=26):
    g = torch.Generator("cuda").manual_seed(seed)
    return pipe(prompt=prompt, negative_prompt=NEG, width=w, height=h, num_inference_steps=steps, guidance_scale=5.5, generator=g).images[0]
def cut(img, size):
    o = remove(img, session=CUT).convert("RGBA"); bb = o.getbbox()
    if bb: o = o.crop(bb)
    o.thumbnail((size, size), Image.LANCZOS); return o
made = []
def save(img, name, q=82):
    img.save(f"/kaggle/working/{name}.webp", "WEBP", quality=q, method=6); made.append((name, img)); print("made", name, img.size)

# Leave REDO empty to make everything. Otherwise only these are made, with three tries each.
REDO = {"i_chicken", "i_fishslice", "i_prawn", "i_lettuce", "i_potato", "i_youtiao", "i_glassnoodle", "i_shrimppaste", "broth_tomato"}
TRIES = 3 if REDO else 1


In [ ]:
PHOTO = ", professional food photography, studio softbox light, sharp focus, photorealistic"
POTS = {
 "pot_copper": "empty round hammered copper hot pot with a polished brass rim and two ring handles, clean empty shiny interior, seen from a 35 degree high angle, isolated on a pure white background, product photo",
 "pot_iron": "empty round matte black enamel cast iron hot pot with a thin gold rim and two black ring handles, clean empty interior, seen from a 35 degree high angle, isolated on a pure white background, product photo",
}
for name, p in POTS.items():
    if REDO and name not in REDO: continue
    for k, seed in enumerate([21, 58]):
        t = time.time(); img = gen(p + PHOTO, 1152, 896, seed)
        save(cut(img, 900), f"{name}_{k}"); print(round(time.time() - t), "s")


In [ ]:
BROTHS = {
 "broth_tomato": "top-down close-up of the surface of a tomato hot pot soup, the soup fills the whole picture edge to edge, no bowl, no pot rim visible, glossy orange red soup, floating tomato wedges, tiny oil droplets",
 "broth_mala": "overhead close-up of Sichuan mala hot pot broth filling the entire frame, deep red chili oil surface, floating dried red chilies and Sichuan peppercorns, glossy",
 "broth_mushroom": "overhead close-up of mushroom hot pot broth filling the entire frame, clear golden brown soup surface, floating sliced mushrooms, small oil droplets",
 "broth_chicken": "overhead close-up of herbal chicken hot pot broth filling the entire frame, clear golden soup surface, goji berries, red dates, ginger slices",
 "broth_pork": "overhead close-up of creamy pork bone hot pot broth filling the entire frame, milky white soup surface, chopped spring onion",
 "broth_tomyum": "overhead close-up of tom yum hot pot broth filling the entire frame, orange soup surface, lemongrass, kaffir lime leaves, red chili slices",
}
for name, p in BROTHS.items():
    if REDO and name not in REDO: continue
    for t in range(TRIES):
        img = gen(p + PHOTO, 896, 896, 7 + 31 * t); img = img.resize((560, 560), Image.LANCZOS); save(img, name if TRIES == 1 else f"{name}_try{t}", 80)


In [ ]:
ITEMS = {
 "i_beef": "rolled thin slices of raw marbled wagyu beef for hot pot",
 "i_leanbeef": "rolled thin slices of raw lean red beef for hot pot",
 "i_lamb": "rolled thin slices of raw lamb for hot pot",
 "i_pork": "thin slices of raw pork for hot pot, pink with white fat",
 "i_chicken": "a neat stack of five thin slices of raw pink chicken breast meat for hot pot, clearly visible meat texture",
 "i_luncheon": "three slices of pink luncheon meat",
 "i_prawn": "three whole cooked prawns with bright orange red shells and tails, curled",
 "i_fishslice": "five thin slices of raw white fish fillet with pale pink edges, fanned out, glistening",
 "i_fishball": "three white Asian fish balls",
 "i_fishtofu": "three golden fried fish tofu pieces",
 "i_beefball": "three brown Chinese beef balls",
 "i_crab": "three red and white crab sticks",
 "i_shrimppaste": "a small mound of smooth light pink minced shrimp paste for hot pot, no meat slices, no bowl",
 "i_dumpling": "three Chinese pork wontons",
 "i_tofu": "three cubes of soft white tofu",
 "i_tofupuff": "three golden fried tofu puffs",
 "i_yuba": "dried beancurd skin sheets, yuba",
 "i_egg": "one boiled egg cut in half showing the yolk",
 "i_quail": "three peeled quail eggs",
 "i_lettuce": "a small head of crisp bright green romaine lettuce leaves, vivid green",
 "i_napa": "a few napa cabbage leaves",
 "i_spinach": "a small bunch of fresh spinach",
 "i_enoki": "a bunch of fresh enoki mushrooms",
 "i_shiitake": "three fresh shiitake mushrooms with cross cut caps",
 "i_corn": "two short pieces of sweet corn on the cob",
 "i_wintermelon": "cubes of peeled winter melon",
 "i_potato": "six thick round slices of peeled raw yellow potato, fanned out, solid and opaque",
 "i_lotus": "three slices of lotus root",
 "i_kelp": "a few knotted kelp seaweed strips",
 "i_glassnoodle": "a small bundle of soaked translucent glass noodles, mung bean vermicelli, clear and shiny, light grey white",
 "i_noodle": "a small nest of fresh hand-pulled noodles",
 "i_udon": "a small nest of thick udon noodles",
 "i_instant": "a block of dried instant noodles",
 "i_youtiao": "a long golden brown Chinese fried dough stick, youtiao, two strips joined side by side, puffy and crispy, cut into four short pieces",
}
DARK = {"i_chicken", "i_fishslice", "i_glassnoodle", "i_shrimppaste", "i_potato"}   # pale food cuts out cleaner from a dark background
for k, (name, p) in enumerate(ITEMS.items()):
    if REDO and name not in REDO: continue
    bg = ", isolated on a plain dark charcoal grey background" if name in DARK else ", isolated on a pure white background"
    for t in range(TRIES):
        img = gen(p + bg + ", seen from a 45 degree angle" + PHOTO, 768, 768, 100 + k + 977 * t, 26)
        save(cut(img, 260), name if TRIES == 1 else f"{name}_try{t}")


In [ ]:
# a contact sheet so the results can be checked quickly
W = 6; cells = [(n, i) for n, i in made]; rows = (len(cells) + W - 1) // W
sheet = Image.new("RGB", (W * 220, rows * 240), "#fbefe9"); d = ImageDraw.Draw(sheet)
for k, (n, i) in enumerate(cells):
    t = i.copy().convert("RGBA"); t.thumbnail((200, 200)); x, y = (k % W) * 220 + 10, (k // W) * 240 + 6
    sheet.paste(t, (x + (200 - t.width) // 2, y + (200 - t.height) // 2), t); d.text((x, y + 208), n, fill="#5a2433")
sheet.save("/kaggle/working/contact.jpg", quality=80); print(len(made), "images made")
